Propósito: Filtra las empresas del SPY500 para traer los papeles en sobreventa. Usa parametros de los indicadores RSI, MACD y Koncorde en la temporalidad de 60min y 7días.

Este filtro busca capturar movimientos que duran entre 2 a 5 días.

In [1]:
%pip install -q pandas-ta yfinance lxml

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Importa las librerias requeridas.
import pandas as pd
import requests
import io
import yfinance as yf
import pandas_ta as ta

In [3]:
#Scrapea Wikipedia para acceder a los nombres de los tickers del S&P500.

#1.1 Define la URL y un 'User-Agent' para evitar el bloqueo.
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
header = {
  "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/50.0.2661.75 Safari/537.36",
  "X-Requested-With": "XMLHttpRequest"
}

#1.2 Realiza la petición con el header de seguridad
r = requests.get(url, headers=header)

#1.3 Lee la tabla desde el texto de la respuesta, usando io.StringIO para forzar la interpretación como string HTML
tables = pd.read_html(io.StringIO(r.text))
df_spy = tables[0]

#1.4 Extrae la lista de Tickers
tickers_a_analizar = [t.replace('.', '-') for t in df_spy['Symbol'].tolist()]

#1.5 Añade tickers de interes.

mis_tickers = ['^VIX', 'SPY', 'SPXL' , 'TQQQ' , 'QQQ' , 'URA' , 'XLI' , 'XLF' , 'XLRE' , 'XLB' , 'XLY' , 'XLE', 'XLV' ,
                'XBI' , 'XLC' , 'BTC-USD' , 'DIA']

tickers_a_analizar.extend(mis_tickers)

print(f"Se han cargado {len(tickers_a_analizar)} tickers...")

Se han cargado 520 tickers...


In [4]:
#DESCARGA DE DATOS TEMPORALIDAD 60 MINUTOS.

tickers_limpios = tickers_a_analizar

print(f"Descargando datos para {len(tickers_limpios)} activos...")
data_60m = yf.download(tickers_limpios, period="2mo", interval="60m", group_by='ticker', progress=False)

Descargando datos para 520 activos...


In [5]:
#ANALISIS TEMPORALIDAD 60MIN.

empresas_60min = []

for ticker in tickers_limpios:
    try:
        df = data_60m[ticker].dropna()
        if len(df) < 50: continue

        # Cálculos Técnicos
        df.ta.rsi(length=10, append=True)
        df['RSI_MA'] = ta.sma(df['RSI_10'], length=14)
        df.ta.macd(fast=12, slow=26, signal=9, append=True)

        # Variables para validación.
        rsi_act = df['RSI_10'].iloc[-1]
        rsi_ant = df['RSI_10'].iloc[-2]
        macd_act = df['MACD_12_26_9'].iloc[-1]
        sig_act = df['MACDs_12_26_9'].iloc[-1]

        # Lógica de Filtrado 60min
        if (rsi_act <= 35 
            and macd_act <= sig_act 
                ):
            
            empresas_60min.append({
                'Ticker': ticker,
                'RSI actual': round(rsi_act, 1),
                'RSI anterior': round(rsi_ant, 1),
                #Añade el MACD.
                'MACD': round(macd_act, 3)
                
            })
    except Exception as e:
      print(f"Error en {ticker}: {e}")

print("\n" + "="*95)
if empresas_60min:
    df_final = pd.DataFrame(empresas_60min)
    print(df_final.sort_values(by='RSI actual').to_string(index=False))
else:
    print("Sin señales detectadas.")


 Ticker  RSI actual  RSI anterior    MACD
   PANW        18.2          19.4 -13.707
   NTAP        19.1          21.7  -6.733
   TQQQ        19.9          20.7  -2.397
   PLTR        19.9          20.3  -4.537
    QQQ        20.0          20.7  -7.601
   CRWD        20.2          20.5  -8.723
    HPE        20.2          21.3  -3.240
      P        20.9          23.0  -6.243
   INTC        21.9          23.5  -3.066
   KLAC        21.9          25.3  -4.806
   CSCO        21.9          23.7  -1.876
    STX        22.0          24.1 -26.481
   ANET        22.0          24.5  -6.691
   AVGO        22.2          24.9  -6.203
   FTNT        22.4          22.6  -5.334
    UAL        22.4          23.4  -1.571
   SMCI        22.4          23.7  -1.866
   MRVL        24.2          27.6  -8.162
    WDC        24.3          28.8 -11.443
    AMD        25.1          28.4 -15.143
   DDOG        25.8          26.2  -8.597
  GOOGL        26.1          24.6  -2.780
   TWLO        26.2          23.6

In [6]:
# DESCARGA DATOS DIARIOS
print(f"Descargando datos DIARIOS para {len(tickers_limpios)} activos...")
data_1d = yf.download(tickers_limpios, period="2mo", interval="1d", group_by='ticker', progress=False)


Descargando datos DIARIOS para 520 activos...


In [7]:
# ANÁLISIS DIARIO

empresas_diario = []

for ticker in tickers_limpios:
    try:
        df = data_1d[ticker].dropna()
        if len(df) < 30: continue

        # Cálculos Técnicos.
        df.ta.rsi(length=14, append=True)
        df.ta.macd(fast=12, slow=26, signal=9, append=True)

        # Variables de validación
        rsi_ant = df['RSI_14'].iloc[-2]
        rsi_act = df['RSI_14'].iloc[-1]
        macd_act = df['MACD_12_26_9'].iloc[-1]
        sig_act = df['MACDs_12_26_9'].iloc[-1]
        

        precio_act = df['Close'].iloc[-1]

        # LÓGICA DE FILTRADO DIARIO
        estuvo_sobrevendido = rsi_act <= 35 or rsi_ant <= 35
        dejo_de_caer = rsi_act >= rsi_ant

        if estuvo_sobrevendido and dejo_de_caer:
            
            empresas_diario.append({
                'Ticker': ticker,
                #'Prec diario': round(precio_act, 2),
                'RSI actual': round(rsi_act, 1),
                'RSI anterior': round(rsi_ant, 1),
                'MACD': round(macd_act, 3),
                #'MACD_SIGNAL': round(sig_act , 3)
            })
    except:
        continue

print("\n" + "="*95)


print("PAPELES DETECTADOS EN DIARIO:")
if empresas_diario:
    df_final = pd.DataFrame(empresas_diario)
    print(df_final.sort_values(by='RSI actual').to_string(index=False))
else:
    print("Sin señales diarias.")


PAPELES DETECTADOS EN DIARIO:
Ticker  RSI actual  RSI anterior   MACD
  NTRS        32.6          32.3 -4.265
   PNC        35.4          27.8 -5.119
   KEY        36.1          34.7 -0.384
   FRT        36.6          31.9 -2.019
     C        36.9          33.9 -2.405
   BNY        38.3          34.5 -4.293
  FDXF        38.5          31.8 -3.914
   BKR        41.1          33.4 -1.364
   SLB        41.6          33.4 -1.357


In [ ]:
#TICKERS REPETIDOS EN AMBAS TEMPORALIDADES.

tickers_60 = [e['Ticker'] for e in empresas_60min]
ticker_1d = [e['Ticker'] for e in empresas_diario]

tickers_comunes = list(set(tickers_60) & set(ticker_1d)) # Interseccion de los elementos en la lista.

print(tickers_comunes)


['AMD', 'ALB', 'GOOGL', 'GOOG', 'AMZN', 'APH', 'ADI', 'APO', 'ANET', 'AXON', 'AVGO', 'BG', 'SCHW', 'CMG', 'CSCO', 'CRWD', 'DDOG', 'DAL', 'ETN', 'P', 'FFIV', 'FSLR', 'FTNT', 'HPE', 'HUM', 'INTC', 'IBKR', 'JBL', 'KLAC', 'MRVL', 'META', 'MGM', 'NTAP', 'ORCL', 'PLTR', 'PANW', 'PLD', 'PTC', 'HOOD', 'STX', 'LUV', 'SMCI', 'SNPS', 'TWLO', 'UAL', 'VYLR', 'WDC', 'SPY', 'SPXL', 'TQQQ', 'QQQ', 'XBI', 'XLC', 'BTC-USD']
['BKR', 'BNY', 'C', 'FRT', 'FDXF', 'KEY', 'NTRS', 'PNC', 'SLB', 'GOOGL']
['GOOGL']


In [9]:
# DESCARGA DATOS SEMANAL.
print(f"Descargando datos SEMANALES para {len(tickers_limpios)} activos...")
data_7d = yf.download(tickers_limpios, period="1y", interval="1wk", group_by='ticker', progress=False)

Descargando datos SEMANALES para 520 activos...


In [10]:
# ANÁLISIS SEMANAL

empresas_semanal = []

for ticker in tickers_limpios:
    try:
        df = data_7d[ticker].dropna()
        if len(df) < 50: continue

        # Cálculos Técnicos.
        df.ta.rsi(length=10, append=True)
        df.ta.macd(fast=12, slow=26, signal=9, append=True)

        # Variables de validación
        rsi_act = df['RSI_10'].iloc[-1]
        macd_act = df['MACD_12_26_9'].iloc[-1]
        sig_act = df['MACDs_12_26_9'].iloc[-1]
        rsi_ant = df['RSI_10'].iloc[-2]

        # Lógica de filtrado.
        if rsi_act <= 40 and macd_act <= sig_act:
        
            empresas_semanal.append({
                'Ticker': ticker,
                'RSI_Act': round(rsi_act, 1),
                'RSI_Ant': round(rsi_ant, 1),
                'MACD_S': round(macd_act, 3),
                'MACD_S_SIGNAL': round(sig_act , 3)
            })
    except:
        continue

print("\n" + "="*95)


print("PAPELES DETECTADOS EN SEMANAL.")
if empresas_semanal:
    df_final = pd.DataFrame(empresas_semanal)
    print(df_final.sort_values(by='RSI_Act').to_string(index=False))
else:
    print("Sin señales semanales.")


PAPELES DETECTADOS EN SEMANAL.
Ticker  RSI_Act  RSI_Ant   MACD_S  MACD_S_SIGNAL
   MGM     16.5     17.2   -2.373         -0.224
  WYNN     22.5     21.3   -7.923         -6.059
   PNR     24.3     20.6   -8.638         -8.473
  SKYD     26.4     40.5   -0.597         -0.530
   URA     26.6     33.1   -2.382         -1.749
    GS     27.6     29.6   -6.457         17.576
    MS     27.7     34.1    0.340          5.440
   ALB     27.8     30.2  -13.064         -9.231
   LII     28.3     27.8  -40.476        -30.870
   PLD     28.6     33.7   -1.606          0.492
   APP     29.0     30.4  -61.626        -55.906
   HWM     29.5     31.4   -4.835          2.849
   CRH     29.7     23.9   -7.450         -6.304
   BAC     30.5     37.4    0.226          1.511
   PCG     30.6     30.8   -1.002         -0.532
   FRT     30.9     30.3   -0.578          1.648
    BX     31.1     36.8   -3.349         -1.150
   LVS     31.3     21.9   -4.127         -3.574
   KKR     31.7     36.5   -3.302    